# Challenge One: Real-Time Weather Alerts Agent

**Goal:** Demonstrate the ability to create and test an agent using the Google Agent Development Kit (ADK).

**Requirements covered in this notebook:**
1. An agent that uses tools to retrieve real-time weather data for user locations.
2. A weather summary / alert based on current conditions and location.
3. Test code demonstrating the agent works for multiple US cities.
4. Support for both a Gemini model and a third-party model (OpenAI GPT via LiteLLM).




In [1]:
# 1. Install dependencies
!pip install --upgrade --quiet google-adk google-cloud-aiplatform litellm requests openai


In [2]:
# 2. Imports and configuration
import os
import requests
from typing import Optional, List, Dict, Tuple

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

# --- Configuration ---
# Set these as environment variables (recommended) or fill in directly.
# GOOGLE_MAPS_API_KEY: Google Maps Platform API key with the Geocoding API enabled.
# GOOGLE_API_KEY / GOOGLE_CLOUD_PROJECT: used by ADK/Vertex for the Gemini model.
# OPENAI_API_KEY: used by LiteLLM to call GPT as the third-party model.

GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_GOOGLE_MAPS_API_KEY")
os.environ.setdefault("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")

MODEL_GEMINI_FLASH = "gemini-2.5-flash"
MODEL_GPT = "openai/gpt-4o"


In [3]:
# 3. Tool: convert a place name to latitude/longitude using the Google Maps Geocoding API
def get_lat_lon(place: str) -> Optional[Tuple[float, float]]:
    """
    Convert a place name (e.g. a city and state) into geographic coordinates
    using the Google Maps Geocoding API.

    Args:
        place (str): A human-readable location, e.g. "Chicago, IL" or
            "1600 Amphitheatre Parkway, Mountain View, CA".

    Returns:
        Optional[Tuple[float, float]]: A (latitude, longitude) tuple, or
        None if the location could not be geocoded or an error occurred.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        location = data["results"][0]["geometry"]["location"]
        return (location["lat"], location["lng"])
    except (requests.RequestException, KeyError, IndexError):
        return None


In [4]:
# 4. Tool: fetch the extended weather forecast from the National Weather Service API
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
        each containing keys such as "name", "temperature", "temperatureUnit",
        "windSpeed", "windDirection", "shortForecast", and "detailedForecast".
        Returns None if data is unavailable or an error occurs.
    """
    headers = {"User-Agent": "adk-weather-alerts-agent (contact: akhil.sharma@wwt.com)"}

    try:
        # Step 1: resolve the lat/lon to a NWS gridpoint / forecast URL.
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        # Step 2: fetch the extended forecast periods.
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        return [
            {
                "name": p.get("name", ""),
                "temperature": str(p.get("temperature", "")),
                "temperatureUnit": p.get("temperatureUnit", ""),
                "windSpeed": p.get("windSpeed", ""),
                "windDirection": p.get("windDirection", ""),
                "shortForecast": p.get("shortForecast", ""),
                "detailedForecast": p.get("detailedForecast", ""),
            }
            for p in periods
        ]
    except (requests.RequestException, KeyError, IndexError):
        return None


## Building the agent

The agent below is given both tools (`get_lat_lon` and `get_extended_weather_forecast`) and
instructions that tell it how to combine them: geocode the requested place, pull the forecast,
then summarize current conditions and flag anything alert-worthy (severe heat/cold, high wind,
storms, etc.).


In [5]:
# 5. Agent instructions
WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a friendly and knowledgeable weather alerts assistant for locations in the
United States.

When a user asks about the weather for a place:
1. Use the `get_lat_lon` tool to convert the place name into latitude/longitude.
   If it fails, tell the user you could not find that location and ask them to clarify
   (e.g. add a state).
2. Use the `get_extended_weather_forecast` tool with those coordinates to retrieve the
   forecast periods.
3. Summarize the current/upcoming conditions in plain language: temperature, wind, and
   general outlook.
4. Proactively call out anything alert-worthy: extreme heat (>= 95F) or cold (<= 20F),
   high winds (>= 25 mph), or forecasts mentioning storms, tornadoes, snow, or ice. If
   nothing stands out, say conditions look normal.
5. Keep responses concise and easy to scan, and always name the city/location you are
   reporting on.
"""


In [6]:
# 6. Agent variant 1: Gemini model
weather_agent_gemini = Agent(
    name="Pat",
    model=MODEL_GEMINI_FLASH,
    description="Pat the Friendly Weather Agent (Gemini).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],
)

# 7. Agent variant 2: third-party model (OpenAI GPT) via LiteLLM
weather_agent_gpt = Agent(
    name="PatGPT",
    model=LiteLlm(model=MODEL_GPT),
    description="Pat the Friendly Weather Agent (GPT).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],
)


## Running the agents

Each agent is hosted in a small `AdkApp`, given a session, and then queried. The helper
function below wraps the three steps shown in the workshop slides (create app -> create
session -> query) so we can reuse it for every city and every model.


In [7]:
# 8. Helper to run a query against an agent and print the response
from vertexai.preview import reasoning_engines
from IPython.display import Markdown, display

def ask_agent(agent: Agent, question: str, user_id: str = "test-user-id") -> str:
    """
    Host the given agent in an AdkApp, create a session, and query it once.

    Args:
        agent (Agent): The ADK agent to run.
        question (str): The natural-language question/prompt to send.
        user_id (str): An identifier for the querying user/session owner.

    Returns:
        str: The text of the agent's final response.
    """
    app = reasoning_engines.AdkApp(agent=agent)
    session = app.create_session(user_id=user_id)

    # Depending on the installed google-cloud-aiplatform version,
    # create_session() returns either an object with an `.id` attribute
    # or a plain dict with an "id" key. Handle both.
    session_id = session["id"] if isinstance(session, dict) else session.id

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=question,
        ):
            last_event = event
    except Exception as e:
        return f"Error while querying agent '{agent.name}': {e}"

    # A failed model/tool call sometimes surfaces as an event without a
    # "content" key (e.g. a rate-limit or auth error) rather than a raised
    # exception. Surface that clearly instead of crashing on a KeyError.
    if not last_event or "content" not in last_event:
        return f"Agent '{agent.name}' did not return a valid response. Raw event: {last_event}"

    return last_event["content"]["parts"][0]["text"]


In [8]:
# 9. Test code: run the Gemini-backed agent against multiple US cities
test_cities = [
    "New York, NY",
    "Los Angeles, CA",
    "Chicago, IL",
    "Miami, FL",
    "Seattle, WA",
]

print("=== Gemini agent ===")
for city in test_cities:
    print(f"\n--- {city} ---")
    response = ask_agent(weather_agent_gemini, f"What's the weather like in {city}? Any alerts I should know about?")
    display(Markdown(response))


=== Gemini agent ===

--- New York, NY ---


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
/usr/local/lib/python3.12/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


Here's the weather forecast for New York, NY:

Today, it will be sunny with a high near 81°F and a west wind of 2 to 9 mph.

**Alerts:**
Be aware that showers and thunderstorms are likely on Thursday and Thursday night.
*   **Thursday:** A chance of rain showers between 8 AM and 11 AM, then showers and thunderstorms likely. Partly sunny, with a high near 81°F and a south wind of 7 to 15 mph. Chance of precipitation is 70%. New rainfall amounts between a quarter and half of an inch possible.
*   **Thursday Night:** Showers and thunderstorms likely. Mostly cloudy with a low around 72°F and a southwest wind of 3 to 14 mph. Chance of precipitation is 60%. New rainfall amounts between a tenth and quarter of an inch possible.

For the rest of the week, conditions look normal with temperatures in the low 80s and mostly sunny skies, though there's a slight chance of rain showers and thunderstorms on Friday and Tuesday. Winds are expected to remain below 25 mph.


--- Los Angeles, CA ---


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


...

Here's the weather for Chicago, IL:

Today, expect a high near 84°F with mostly sunny skies, though there's a slight chance of showers and thunderstorms after 3 PM. Winds will be from the west-southwest around 15 mph, with gusts as high as 25 mph. Tonight, the low will be around 67°F, with a slight chance of showers and thunderstorms before 7 PM, then mostly clear skies. Winds will be from the west-northwest at 5 to 10 mph, with gusts up to 20 mph.

**Alerts:** Winds today could gust up to 25 mph, which is on the higher side. There's also a slight chance of thunderstorms today and tonight.


--- Miami, FL ---


Here's the weather forecast for Miami, FL:

**Today:** Expect a high near 89°F with a heat index as high as 103°F. Winds will be from the southeast at 8 to 12 mph. There's a 50% chance of showers and thunderstorms.

**Tonight:** The low will be around 83°F with an east wind around 10 mph. There's a 40% chance of showers and thunderstorms.

**Thursday:** Look for a high near 88°F with a heat index as high as 104°F and a southeast wind around 10 mph. There's a 50% chance of showers and thunderstorms.

**Alerts:**
*   **Extreme Heat:** The heat index is expected to reach 103°F today and 104°F on Thursday. Please take precautions against the heat.
*   **Thunderstorms:** There is a chance of showers and thunderstorms throughout the forecast period, with an increased likelihood over the weekend.


--- Seattle, WA ---


Here's the weather for Seattle, WA:

**Today:** Mostly sunny with a high near 79°F, and a south-southwest wind around 6 mph.
**Tonight:** Partly cloudy with a low around 59°F, and a south-southwest wind of 3 to 7 mph.
**Thursday:** Partly sunny with a high near 74°F, and a south-southwest wind of 7 to 10 mph.

Conditions look normal. There is a chance of rain beginning Friday evening and continuing off and on through next Tuesday, but no severe weather is expected.

In [9]:
# 10. Test code: run the GPT-backed agent against the same cities to confirm
# the agent also works with a non-Gemini, third-party model.
print("=== GPT agent (via LiteLLM) ===")
for city in test_cities:
    print(f"\n--- {city} ---")
    response = ask_agent(weather_agent_gpt, f"What's the weather like in {city}? Any alerts I should know about?")
    display(Markdown(response))


=== GPT agent (via LiteLLM) ===

--- New York, NY ---


In New York, NY, the weather today is sunny with a high of 81°F and a southwest wind blowing at 2 to 9 mph. 

**Alert-worthy conditions**: For Thursday, expect showers and thunderstorms likely, especially from the morning through the evening, with a 70% chance of precipitation and south winds 7 to 15 mph. Another round of showers and storms is probable Thursday night. Be prepared for potential rain and storms.

Otherwise, apart from Thursday's weather, conditions look relatively normal with temperatures in the low 80s and mild wind speeds over the upcoming days.


--- Los Angeles, CA ---


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


For Los Angeles, CA:

- **Today**: It's quite hot with temperatures reaching up to 99°F. Expect sunny skies with a south-southwest wind at 0 to 10 mph. This qualifies as extreme heat.
- **Tonight**: Mostly cloudy with a low of 74°F and south winds at 0 to 10 mph.
- **Thursday**: Partly sunny with a high near 98°F and south winds from 0 to 10 mph.
- **Friday**: Temperatures rise again to around 99°F with mostly sunny skies and southwest winds at 0 to 10 mph.
- **Saturday**: Still warm at 93°F under sunny skies, with a south-southwest breeze.
- **Sunday & Monday**: More moderate temperatures, between 85°F and 88°F, sunny with low wind.

**Alerts**: 
- Extreme heat for the next couple of days with highs near 99°F, so stay hydrated and keep cool.

Conditions look mostly normal beyond the heat. Take care!

...

The weather in Chicago, IL today is mostly sunny with a slight chance of showers and thunderstorms later in the afternoon. The high temperature is around 84°F, with winds from the west-southwest at 15 mph, gusting up to 25 mph. Tonight, it will be mostly clear with a low of 67°F.

No extreme temperatures or significant weather alerts are in the forecast for the next few days. Conditions seem normal overall, but do watch out for those brief chances of thunderstorms today.


--- Miami, FL ---


In Miami, FL, the weather today and the coming days will be quite warm and humid with high chances of showers and thunderstorms. Here's a quick summary:

- **Today**: High of 89°F, with a heat index up to 103°F. Southeast winds 8 to 12 mph. 50% chance of showers and thunderstorms.
- **Tonight**: Low of 83°F, with east winds around 10 mph. 40% chance of showers and thunderstorms.
- **Thursday**: High of 88°F, with a heat index up to 104°F. Southeast winds around 10 mph. 50% chance of showers and thunderstorms.
  
Expect similar conditions through the weekend, with temperatures peaking around 90°F and a consistent chance for showers and thunderstorms.

**Alert**: The high heat index values and potential for thunderstorms make it advisable to stay hydrated and remain informed about changing weather conditions. If you're outdoors, take precautions to stay cool and keep an eye on the sky for storm developments.


--- Seattle, WA ---


In Seattle, WA, today the weather is mostly sunny with a high near 79°F and a light south southwest wind at 6 mph. Tonight will be partly cloudy with temperatures dropping to around 59°F, and a gentle wind from the same direction.

Over the next few days, the weather will generally be mild. Thursday and Friday's highs will be in the mid-70s with partly sunny skies. There's a chance of light rain on Friday evening extending into Saturday morning.

No extreme weather alerts at the moment—conditions look normal for Seattle! Let me know if you have any other questions about the weather.